In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class LayerNormalization(nn.Module):
    def __init__(self, model_dim):
        super().__init__()
        self.model_dim = model_dim
        self.gamma = nn.Parameter(torch.ones(self.model_dim))
        self.beta = nn.Parameter(torch.zeros(self.model_dim))
        self.eps = 1e-5

    def forward(self, x):
        B, T, D = x.shape
        x_centered = x - torch.mean(x, dim=-1, keepdim=True)
        x_var = torch.mean(x_centered**2, dim=-1, keepdim=True)
        output = x_centered/torch.sqrt(x_var + self.eps)
        return self.gamma * output + self.beta 
    
class Linear(nn.Module):
    def __init__(self, in_dim, out_dim, bias=False):
        super().__init__()

        #define w parameter, no bias for now
        #divide by math.sqrt(in_dim) to avoid bad parameter initialization
        self.W = nn.Parameter(torch.randn(in_dim, out_dim)/ math.sqrt(in_dim))
    
    def forward(self, x):
        return torch.matmul(x, self.W)

class GLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        B, T, D1 = x.shape
        N = D1 // 2
        l_proj1 =  x[:,:, :N]
        l_proj2 = x[:, :, N:]

        return l_proj1 * torch.sigmoid(l_proj2)

def SoftMax(x, dim):

        #numerically stable softmax
        x_max, _ = torch.max(x, dim=dim, keepdim=True)       #values, indices
        x_scaled = x - x_max
        num = torch.exp(x_scaled)
        den = torch.sum(num, dim=dim, keepdim=True)
        return num / den

def CrossEntropy(logits, targets, ignore_index=-100):

    #skip logits with ignore targets in the input
    mask = targets != ignore_index
    logits = logits[mask]                                                                                                            
    targets = targets[mask]

    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    predictions = log_probs[torch.arange(len(targets), device=logits.device), targets]
    loss = -predictions.mean()
    
    return loss

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, model_dim, num_heads, device):
        super().__init__()
        self.model_dim = model_dim
        self.head_dim = model_dim // num_heads
        self.num_heads = num_heads
        self.device = device

        self.q_proj = Linear(self.model_dim, self.model_dim, bias=False)
        self.k_proj = Linear(self.model_dim, self.model_dim, bias=False)
        self.v_proj = Linear(self.model_dim, self.model_dim, bias=False)

        self.o_proj = Linear(self.model_dim, self.model_dim, bias=False)
    

    def forward(self, x, causal=True, attention_mask=None):
        B, T, D = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        score = torch.matmul(q, k.transpose(-2, -1)) #B, H, T, T
        score = score/math.sqrt(self.head_dim)

        if causal:
            if not attention_mask:
                attention_mask = torch.tril(torch.ones(T, T)).to(self.device)

            score = score.masked_fill(attention_mask==0, float('-inf'))
            
        attn_score = SoftMax(score, dim=-1) #B, H, T, T
        attention = torch.matmul(attn_score, v) #B, H, T, D_h
        attention = attention.transpose(1, 2).contiguous().view(B, T, D) #B, T, D

        return self.o_proj(attention) #B, T, D

class LayerNormalization(nn.Module):
    def __init__(self, model_dim):
        super().__init__()
        self.model_dim = model_dim
        self.gamma = nn.Parameter(torch.ones(self.model_dim))
        self.beta = nn.Parameter(torch.zeros(self.model_dim))
        self.eps = 1e-5

    def forward(self, x):
        B, T, D = x.shape
        x_centered = x - torch.mean(x, dim=-1, keepdim=True)
        x_var = torch.mean(x_centered**2, dim=-1, keepdim=True)
        output = x_centered/torch.sqrt(x_var + self.eps)
        return self.gamma * output + self.beta 

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, model_dim, num_heads, device) :
        super().__init__()

        self.MHSA = MultiHeadAttention(model_dim, num_heads, device)
        self.layernorm1 = LayerNormalization(model_dim)
        self.layernorm2 = LayerNormalization(model_dim)

        self.FFN = nn.Sequential(   
                        Linear(model_dim, 4*model_dim, bias=False),
                        GLU(),
                        Linear(2*model_dim, model_dim, bias=False)
                    )
    def forward(self, x, attention_mask=None):
        x1 = x + self.MHSA(self.layernorm1(x), attention_mask)
        x2 = x1 + self.FFN(self.layernorm2(x1))
        return x2

In [ ]:
class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.model_dim = cfg.model_dim
        self.num_heads = cfg.num_heads
        self.num_blocks = cfg.num_blocks
        self.vocab_size = cfg.vocab_size
        self.max_seq_len = cfg.max_seq_length
        self.device = cfg.device

        #transformer blocks 
        self.num_blocks = self.num_blocks
        self.blocks = nn.ModuleList([TransformerBlock(self.model_dim, self.num_heads, self.device) for _ in range(self.num_blocks)])
        self.logit_proj = Linear(self.model_dim, cfg.vocab_size, bias=False)

        #token and position embeddings
        self.token_embeddings = nn.Embedding(self.vocab_size, self.model_dim)
        self.pos_emb = nn.Embedding(self.max_seq_len, self.model_dim)


    def input_embeddings(self, x):
        B, T = x.shape
        x1 = self.token_embeddings(x) #B, T -> #B, T, D
        pos_emb = self.pos_emb(torch.arange(T, device=self.device))
        x = x1 + pos_emb #B, T, D + (T, D)->B, T, D
        return x

    def forward(self, x, targets=None, input_embeds=None, attention_mask=None):
        if input_embeds is not None:
            x = input_embeds

        else:
            x = self.input_embeddings(x)
           
        for i in range(self.num_blocks):
            x = self.blocks[i](x, attention_mask)
        
        logits = self.logit_proj(x) #B, T, V

        B, T1, V = logits.shape
        logits = logits.view(B*T1, V)
        targets = targets.reshape(B*T1)

        loss = CrossEntropy(logits, targets)

        return logits, loss

In [ ]:
class ViT(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.model_dim = cfg.model_dim
        self.num_heads = cfg.num_heads
        self.num_blocks = cfg.num_blocks
        self.num_patches = cfg.num_patches # H//P * W //P
        self.device = cfg.device
        self.patch_size = cfg.patch_size #P
        self.patch_dim = cfg.patch_dim   #C*P*P
        self.num_classes = cfg.num_classes

        #transformer blocks (non-causal for ViT — every token attends to every other)
        self.num_blocks = self.num_blocks
        self.blocks = nn.ModuleList([TransformerBlock(self.model_dim, self.num_heads, self.device, causal=False) for _ in range(self.num_blocks)])
        self.logit_proj = Linear(self.model_dim, cfg.num_classes, bias=False)

        #token and position embeddings
        self.cls_token = nn.Parameter(torch.randn(1, 1, self.model_dim))
        self.x_proj = Linear(self.patch_dim, self.model_dim)
        self.pos_emb = nn.Embedding(self.num_patches+1, self.model_dim)

    
    def img_to_patch(self, x):
        B, C, H, W = x.shape
    
        P = self.patch_size

        x = x.unfold(dimension=2, size=P, step=P)
        x = x.unfold(dimension=3, size=P, step=P)     #B, C, H/P, W/P, P, P
        x = x.permute(0, 2, 3, 1, 4, 5)         #B, H/P, W/P, C, P, P

        #reshape to [B, T, D]
        num_patches = H//P * W //P
        patch_dim = C*P*P
        x = x.reshape(B, num_patches, C, P, P).reshape(B, num_patches, patch_dim) #B, T, D
        return self.x_proj(x)
        
    def get_contextualized_embeddings(self, x):
        B, C, H, W = x.shape
        x = self.img_to_patch(x)
        cls_token1 = self.cls_token.expand(B, 1, self.model_dim)
        x = torch.cat([cls_token1, x], dim=1)

        #add position embeddings 
        pos_emb = self.pos_emb(torch.arange(self.num_patches+1, device=x.device)) #T, D
        x = x + pos_emb

        #transformer blocks 
        for i in range(self.num_blocks):
            x = self.blocks[i](x)

        return x

    def encode(self, x):
        x = self.get_contextualized_embeddings(x)
        return x[:, 1:, :]
        
    def forward(self, x, targets=None):
       
        x = self.get_contextualized_embeddings(x)
        cls_logit = x[:, 0, :]  #B, D
        logits = self.logit_proj(cls_logit) #B, C

        loss = 0.0
        if targets != None:
            loss = CrossEntropy(logits, targets)

        return logits, loss

In [ ]:
class VLM(nn.Module):
    def __init__(self, cfg):

        super().__init__()

        self.vision_encoder = cfg.vision_encoder
        self.LLM = cfg.LLM

        self.model_dim = cfg.model_dim
        self.vision_model_dim = cfg.vision_model_dim
        self.vision_proj = Linear(cfg.vision_model_dim, cfg.model_dim)
    
    def forward(self, x_img, x_text, targets=None, attention_mask=None):
        
        B, N, D_img = x_img.shape
        img_logits = self.vision_encoder.encode(x_img)
        img_embeddings = self.vision_proj(img_logits)
        text_embeddings = self.LLM.input_embeddings(x_text)

        llm_embeddings = torch.cat([img_embeddings, text_embeddings], dim=1)

        #prepare targets
        pad = torch.full((B, N), -100, dtype=targets.dtype, device=targets.device)                                                           
        targets = torch.cat([pad, targets], dim=1)   # (B, N+T) 

        return self.LLM(input_embeds=llm_embeddings, targets=targets, attention_mask=attention_mask)

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class ConfigParametersViT:
    patch_size: int
    patch_dim: int
    num_patches: int
    device: str
    model_dim: int=256
    num_heads: int=4
    batch_size: int=64
    num_blocks: int=2

@dataclass
class ConfigParametersLLM:
    vocab_size: int
    device: str
    max_seq_length: int=32
    model_dim: int=64
    num_heads: int=4
    block_size: int=32  #block size to get tokens from the dataset
    batch_size: int=4
    num_blocks: int=2   #number of layers
   

@dataclass
class VLMParameters:
    model_dim: int
    vision_model_dim: int
    vision_encoder: Optional[object] = None
    LLM: Optional[object] = None
    

@dataclass
class OptimParameters:
    lr: float=3e-4
    betas: Tuple=(0.9, 0.95)
    eps: float=1e-8